# Albedo Estimation via Latent Bridge Matching*Carme Corbi, David Serrano-Lozano, Javier Vazquez-Corral, Maria Vanrell &mdash; CIC 2026*This notebook walks through the released two-stage intrinsic decomposition pipeline:given a single RGB image, it estimates the **albedo** (the illumination-invariantsurface color) and the **shading**, and checks that they reproduce the input under theimage formation model `I = A · S`.Before running it, install the package from the repository root:```bashpip install -e ".[demo]"```The weights (2 x 5 GB) are downloaded from the Hugging Face Hub the first time thepipeline is loaded, and cached afterwards. A CUDA GPU with ~11 GB of free memory isrecommended.

## 1. Setup

In [ ]:
import osimport timeimport matplotlib.pyplot as pltimport torchfrom PIL import Imagefrom albedo_lbm import IntrinsicDecomposer, evaluate# Point this at a local directory to use weights you already downloaded.MODEL_DIR = os.environ.get("ALBEDO_LBM_WEIGHTS", "davidserra9/albedo-lbm")DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print("Weights :", MODEL_DIR)print("Device  :", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

In [ ]:
def show(images, titles, height=3.2):    """Display a row of images side by side."""    fig, axes = plt.subplots(1, len(images), figsize=(height * len(images) * 1.33, height))    for ax, image, title in zip(axes, images, titles):        ax.imshow(image)        ax.set_title(title, fontsize=11)        ax.axis("off")    fig.tight_layout()    plt.show()

## 2. Load the pipeline`IntrinsicDecomposer` bundles the two models of the paper:| model | task | conditioned on || --- | --- | --- || **LBM-AID** | RGB &rarr; albedo | shading || **LBM-SID** | RGB &rarr; shading | albedo |Loading both takes about a minute the first time (plus the download).

In [ ]:
pipeline = IntrinsicDecomposer.from_pretrained(    MODEL_DIR,    device=DEVICE,    torch_dtype=torch.bfloat16,)print("Ready.")

## 3. Decompose an imageEach model runs a **single bridge step**, so the whole decomposition is three UNetevaluations: a bootstrap albedo pass, a shading pass, and the final albedo pass.

In [ ]:
image = Image.open("../assets/examples/indoor_studio.png").convert("RGB")start = time.time()result = pipeline(image, num_steps=1, return_intermediate=True)print(f"{image.size[0]}x{image.size[1]} decomposed in {time.time() - start:.2f}s")show(    [image, result.albedo, result.shading, result.reconstruction()],    ["Input", "Albedo", "Shading", "Albedo x Shading"],)

The rightmost image is `albedo · shading`, re-synthesized from the two predictions.Its closeness to the input is what the reconstruction loss optimizes during training,and it is the physical-consistency measure reported in the paper.

In [ ]:
import numpy as nprecon = np.asarray(result.reconstruction(), np.float32) / 255target = np.asarray(image, np.float32) / 255print(f"reconstruction MSE: {np.mean((recon - target) ** 2):.4f}")

## 4. The bootstrap passThe albedo model wants a shading estimate and the shading model wants an albedoestimate. The pipeline breaks the circularity by conditioning a first albedo pass onthe input image itself, then uses that bootstrap albedo to estimate shading, and onlythen produces the final albedo. The refinement is usually subtle but consistent: softshadows and inter-reflections left in the bootstrap pass get pushed into the shading.

In [ ]:
show(    [image, result.bootstrap_albedo, result.albedo],    ["Input", "1. Bootstrap albedo\n(conditioned on the input)", "3. Final albedo\n(conditioned on the shading)"],)

## 5. More bridge stepsLBM is trained on four equally spaced timesteps, so you can trade a little compute fora slightly different sample. One step is the default and is what the paper reports.

In [ ]:
image2 = Image.open("../assets/examples/arap_attic.png").convert("RGB")outputs, titles = [image2], ["Input"]for steps in (1, 2, 4):    start = time.time()    outputs.append(pipeline(image2, num_steps=steps).albedo)    titles.append(f"{steps} step(s) -- {time.time() - start:.2f}s")show(outputs, titles)

## 6. Bring your own shadingIf you already have a shading map (from a renderer, or from ground truth), you can feedit directly and skip the shading model. This is also the cheapest option memory-wise:load the pipeline with `load_shading_model=False` to keep a single model on the GPU.

In [ ]:
# Reuse the shading we just estimated, as a stand-in for an external one.own_shading = result.shadingalbedo = evaluate(pipeline.albedo_model, image, own_shading, num_sampling_steps=1)show([image, own_shading, albedo], ["Input", "Given shading", "Albedo"])

## 7. Batch a folderFor more than a handful of images, the command line script is the convenient entry point:```bashpython scripts/infer.py --input assets/examples --output results \    --save_shading --save_reconstruction```

In [ ]:
for name in ["indoor_desk", "arap_alley"]:    img = Image.open(f"../assets/examples/{name}.png").convert("RGB")    out = pipeline(img)    show([img, out.albedo, out.shading], ["Input", "Albedo", "Shading"])

## LimitationsThe models are trained on synthetic indoor data (InteriorVerse and Hypersim), soexpect occasional color shifts on real photographs, and failures on transparent andmetallic materials. Quality also degrades above roughly 2K resolution, since trainingused 256x256 crops.If you use this work, please cite:```bibtex@inproceedings{corbi2026albedo,  title     = {Albedo Estimation via Latent Bridge Matching},  author    = {Corbi, Carme and Serrano-Lozano, David and Vazquez-Corral, Javier and Vanrell, Maria},  booktitle = {Color and Imaging Conference (CIC)},  year      = {2026}}```